<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/2_Ashok_CUDA_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check GPU and setup
!nvidia-smi | head -5


Mon May 25 07:39:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [2]:
!pip install -q ultralytics

from ultralytics import YOLO
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Download a pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # 'n' = nano (smallest, fastest)

print(f"Model: YOLOv8-nano")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Input size: 640×640")
print(f"Classes: {len(model.names)} ({', '.join(list(model.names.values())[:10])}...)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model: YOLOv8-nano
Parameters: 3,157,200
Input size: 640×640
Classes: 80 (person, bicycle, car, motorcycle, airplane, bus, train, truck, boat, traffic light...)


In [3]:
# Download a test image and run detection
!wget -q https://ultralytics.com/images/bus.jpg -O test_image.jpg

results = model('test_image.jpg')

result = results[0]
print(f"\nDetections found: {len(result.boxes)}")
print(f"\n{'Class':<15} {'Confidence':<12} {'Box (x1,y1,x2,y2)'}")
print("-" * 55)
for box in result.boxes:
    cls = model.names[int(box.cls)]
    conf = float(box.conf)
    coords = box.xyxy[0].tolist()
    print(f"{cls:<15} {conf:<12.2f} [{coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f}]")



image 1/1 /content/test_image.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 168.1ms
Speed: 12.0ms preprocess, 168.1ms inference, 72.5ms postprocess per image at shape (1, 3, 640, 480)

Detections found: 6

Class           Confidence   Box (x1,y1,x2,y2)
-------------------------------------------------------
bus             0.87         [23, 231, 805, 757]
person          0.87         [49, 399, 245, 903]
person          0.85         [669, 392, 810, 877]
person          0.83         [222, 406, 345, 858]
person          0.26         [0, 551, 63, 873]
stop sign       0.26         [0, 254, 33, 325]


In [4]:
# Let's see what happens inside YOLO step by step
import torch
import time

print("=== YOLO Pipeline Breakdown ===\n")

# Step 1: Preprocess (what we'll write in CUDA)
img = Image.open('test_image.jpg')
img_np = np.array(img)
print(f"1. PREPROCESS:")
print(f"   Raw image: {img_np.shape} (H×W×C, uint8, 0-255)")

start = time.time()
# Resize to 640×640, normalize to 0-1, convert HWC→CHW
img_resized = np.array(img.resize((640, 640))).astype(np.float32) / 255.0
img_chw = img_resized.transpose(2, 0, 1)  # HWC → CHW
img_batch = img_chw[np.newaxis, ...]  # add batch dim
preprocess_time = (time.time() - start) * 1000
print(f"   After preprocess: {img_batch.shape} (B×C×H×W, float32, 0-1)")
print(f"   Time: {preprocess_time:.1f} ms (Python/NumPy)")
print(f"   → CUDA kernel can do this in <1ms!\n")

# Step 2: Inference (CNN forward pass)
print(f"2. INFERENCE (CNN backbone + detection head):")
print(f"   YOLOv8-nano: 3.15M parameters")
print(f"   225 layers of Conv+BatchNorm+SiLU")
print(f"   Time: 168 ms (PyTorch)")
print(f"   → TensorRT can do this in ~10-20ms!\n")

# Step 3: Postprocess (NMS — what we'll write in CUDA)
print(f"3. POSTPROCESS (NMS):")
print(f"   Raw detections: ~8400 candidate boxes")
print(f"   After NMS: 6 final boxes (removed duplicates)")
print(f"   Time: 72.5 ms (Python)")
print(f"   → CUDA NMS kernel can do this in <5ms!\n")

print(f"=== What we'll build in CUDA ===")
print(f"   Step 4.3: Preprocessing kernel (resize+normalize+HWC→CHW)")
print(f"   Step 4.5: NMS kernel (remove overlapping boxes)")
print(f"   Step 4.8: TensorRT (optimize inference 168ms → ~15ms)")
print(f"   Goal: 252ms → ~20ms total = 50 FPS (real-time!)")


=== YOLO Pipeline Breakdown ===

1. PREPROCESS:
   Raw image: (1080, 810, 3) (H×W×C, uint8, 0-255)
   After preprocess: (1, 3, 640, 640) (B×C×H×W, float32, 0-1)
   Time: 19.3 ms (Python/NumPy)
   → CUDA kernel can do this in <1ms!

2. INFERENCE (CNN backbone + detection head):
   YOLOv8-nano: 3.15M parameters
   225 layers of Conv+BatchNorm+SiLU
   Time: 168 ms (PyTorch)
   → TensorRT can do this in ~10-20ms!

3. POSTPROCESS (NMS):
   Raw detections: ~8400 candidate boxes
   After NMS: 6 final boxes (removed duplicates)
   Time: 72.5 ms (Python)
   → CUDA NMS kernel can do this in <5ms!

=== What we'll build in CUDA ===
   Step 4.3: Preprocessing kernel (resize+normalize+HWC→CHW)
   Step 4.5: NMS kernel (remove overlapping boxes)
   Step 4.8: TensorRT (optimize inference 168ms → ~15ms)
   Goal: 252ms → ~20ms total = 50 FPS (real-time!)


In [5]:
%%writefile preprocess.cu
#include <stdio.h>
#include <stdlib.h>

// Preprocess kernel: resize + normalize + HWC→CHW in ONE kernel
// Input:  H×W×3 (uint8, 0-255, HWC format)
// Output: 3×640×640 (float32, 0-1, CHW format)
__global__ void preprocess_kernel(unsigned char *input, float *output,
                                   int in_h, int in_w,
                                   int out_h, int out_w) {
    int out_x = blockIdx.x * blockDim.x + threadIdx.x;  // output column
    int out_y = blockIdx.y * blockDim.y + threadIdx.y;  // output row
    int c = blockIdx.z;  // channel (0=R, 1=G, 2=B)

    if (out_x < out_w && out_y < out_h && c < 3) {
        // Nearest-neighbor resize: map output pixel to input pixel
        int in_x = (int)((float)out_x / out_w * in_w);
        int in_y = (int)((float)out_y / out_h * in_h);

        // Clamp to valid range
        if (in_x >= in_w) in_x = in_w - 1;
        if (in_y >= in_h) in_y = in_h - 1;

        // Read from HWC format: input[y][x][c]
        unsigned char pixel = input[in_y * in_w * 3 + in_x * 3 + c];

        // Write to CHW format: output[c][y][x], normalized to 0-1
        output[c * out_h * out_w + out_y * out_w + out_x] = (float)pixel / 255.0f;
    }
}

int main() {
    // Simulate an input image (1080×810×3)
    int in_h = 1080, in_w = 810;
    int out_h = 640, out_w = 640;
    int in_size = in_h * in_w * 3;
    int out_size = 3 * out_h * out_w;

    // Create fake input image (random pixels)
    unsigned char *h_input = (unsigned char*)malloc(in_size);
    for (int i = 0; i < in_size; i++) h_input[i] = rand() % 256;

    float *h_output = (float*)malloc(out_size * sizeof(float));

    // GPU memory
    unsigned char *d_input;
    float *d_output;
    cudaMalloc(&d_input, in_size);
    cudaMalloc(&d_output, out_size * sizeof(float));
    cudaMemcpy(d_input, h_input, in_size, cudaMemcpyHostToDevice);

    // Launch kernel
    dim3 threads(16, 16);
    dim3 blocks((out_w+15)/16, (out_h+15)/16, 3);  // 3 channels

    // Warmup
    preprocess_kernel<<<blocks, threads>>>(d_input, d_output, in_h, in_w, out_h, out_w);
    cudaDeviceSynchronize();

    // Benchmark
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    cudaEventRecord(start);
    for (int i = 0; i < 100; i++)
        preprocess_kernel<<<blocks, threads>>>(d_input, d_output, in_h, in_w, out_h, out_w);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);
    ms /= 100;

    // Verify output
    cudaMemcpy(h_output, d_output, out_size * sizeof(float), cudaMemcpyDeviceToHost);

    printf("=== CUDA Image Preprocessing Kernel ===\n\n");
    printf("Input:  %d×%d×3 (uint8, HWC, 0-255)\n", in_h, in_w);
    printf("Output: 3×%d×%d (float32, CHW, 0-1)\n\n", out_h, out_w);
    printf("Operations (all in ONE kernel):\n");
    printf("  1. Resize: %d×%d → %d×%d (nearest neighbor)\n", in_h, in_w, out_h, out_w);
    printf("  2. Normalize: uint8 [0-255] → float32 [0-1]\n");
    printf("  3. Layout: HWC → CHW (what neural networks expect)\n\n");
    printf("Performance:\n");
    printf("  CUDA kernel:    %.3f ms\n", ms);
    printf("  Python/NumPy:   19.3 ms (from earlier test)\n");
    printf("  Speedup:        %.0fx faster! 🚀\n\n", 19.3/ms);

    printf("GPU launch config:\n");
    printf("  Threads: 16×16 = 256 per block\n");
    printf("  Blocks: %d×%d×3 = %d blocks\n", (out_w+15)/16, (out_h+15)/16, ((out_w+15)/16)*((out_h+15)/16)*3);
    printf("  Total threads: %d\n", ((out_w+15)/16)*((out_h+15)/16)*3*256);
    printf("  Each thread: processes ONE pixel in ONE channel\n\n");

    // Verify values are in range
    float min_val = 1.0f, max_val = 0.0f;
    for (int i = 0; i < out_size; i++) {
        if (h_output[i] < min_val) min_val = h_output[i];
        if (h_output[i] > max_val) max_val = h_output[i];
    }
    printf("Output range: [%.3f, %.3f] ✓ (should be 0-1)\n", min_val, max_val);

    free(h_input); free(h_output);
    cudaFree(d_input); cudaFree(d_output);
    return 0;
}


Writing preprocess.cu


In [ ]:
!nvcc preprocess.cu -o preprocess && ./preprocess
